In [2]:
!pip install anndata==0.8.0

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 26.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [4]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools
import os
import time
import matplotlib.patches as mpatches

In [2]:
sam_one = SAM()
sam_one.load_data('SAM_Allen_Institute_directsub_PVH_08082026.h5ad')

In [3]:
sam_test = SAM()
sam_test.load_data('../../Active_SAM_joined/SAM_Allen_Institute_allhypo_01272026.h5ad')

In [10]:
test_dat = sam_test.adata[sam_test.adata.obs['subclass_id_label'] == '133 PVH-SO-PVa Otp Glut']

In [11]:
sam_one.adata.X

<6390x32285 sparse matrix of type '<class 'numpy.float32'>'
	with 24381670 stored elements in Compressed Sparse Row format>

In [8]:
def csr_equal(A, B):
    if A.shape != B.shape:
        return False
    return (A != B).nnz == 0

In [13]:
csr_equal(test_dat.X,sam_one.adata.X)

True

In [15]:
org_dict = {'mo':'SAM_MO_soupx_cleaned_PVH_from0312205_09082026.h5ad','cj':'SAM_CJ_cleaned_PVH_09082026.h5ad','ac':'SAM_AC_soupx_cleaned_PVH_09082026.h5ad',
               'xt':'SAM_XT_cleaned_PVH_09082026.h5ad','dr':'SAM_DR_cleaned_PVH_09082026.h5ad'}
for org in org_dict:
    sam_two = SAM()
    sam_two.load_data(org_dict[org])
    
    sams = {'mg':sam_one,org:sam_two}

    sm = SAMAP(
        sams,
        f_maps = '../../BLASTMAPPING/maps/active_maps/hypo_proj/')
    sm.run(pairwise=True)
    
    save_samap(sm,'sm_Allen_' + org + '_PVH_090820206.pkl')

Not updating the manifold...
Not updating the manifold...
20578 `mg` gene symbols match between the datasets and the BLAST graph.
18026 `mo` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 8.070161819458008
Correcting data with means. 10.828858137130737
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species mo...
Indegree coarsening
0/1 (0, 7699)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.6472962344876333 
Max A.S. improvement: 0.825396748968795 
Min A.S. improvement: 0.0
Calculating gene-gene correlations in the homology graph...
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 6.435533285140991
Correcting data with means. 9.064668893814087
Expanding neighbourhoods of species mg...
Expanding

In [5]:
org_dict = {'mo':'SAM_MO_soupx_cleaned_PVH_from0312205_09082026.h5ad','cj':'SAM_CJ_cleaned_PVH_09082026.h5ad','ac':'SAM_AC_soupx_cleaned_PVH_09082026.h5ad',
               'xt':'SAM_XT_cleaned_PVH_09082026.h5ad','dr':'SAM_DR_cleaned_PVH_09082026.h5ad'}
for o1, o2 in itertools.combinations(org_dict, 2):
    sam_a = SAM()
    sam_a.load_data(org_dict[o1])

    sam_b = SAM()
    sam_b.load_data(org_dict[o2])

    sams = {o1:sam_a,o2:sam_b}

    sm = SAMAP(
        sams,
        f_maps = '../../BLASTMAPPING/maps/active_maps/hypo_proj/')
    sm.run(pairwise=True)

    save_samap(sm,'sm_'+ o1 +'_' + o2 + '_PVH_090820206.pkl')

Not updating the manifold...
Not updating the manifold...
16926 `mo` gene symbols match between the datasets and the BLAST graph.
14726 `cj` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 1.576169729232788
Correcting data with means. 1.544260025024414
Expanding neighbourhoods of species mo...
Expanding neighbourhoods of species cj...
Indegree coarsening
0/1 (0, 2325)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.4907872114780611 
Max A.S. improvement: 0.5017392544980679 
Min A.S. improvement: 0.0
Calculating gene-gene correlations in the homology graph...
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 1.1629672050476074
Correcting data with means. 1.417726993560791
Expanding neighbourhoods of species mo...
Expandin